# What are the latest mRNA-based cancer treatments?

## Healthcare Agentic RAG with Tavily MCP + Qdrant MCP

This project builds an **Agentic RAG biomedical research assistant**.

It uses:

- **Tavily MCP** for real-time web search and content extraction
- **Qdrant MCP** for vector storage and semantic retrieval
- **OpenAI Agents SDK** for orchestration

```text
Research Question
        ↓
Biomedical Research Agent
        ↓
Tavily MCP
(real-time search + extraction)
        ↓
Evidence evaluation + curation
        ↓
Qdrant MCP
(vector knowledge base)
        ↓
Oncology RAG Agent
        ↓
Agentic retrieval + synthesis
        ↓
Grounded research summary
```

Main focus: personalized mRNA neoantigen cancer vaccines, especially melanoma and pancreatic cancer.

> Educational and research use only. Not medical advice.


## 1. Setup

For Windows + VS Code/Jupyter.

Add these to `.env`:

```text
OPENAI_API_KEY=your_openai_key
TAVILY_API_KEY=your_tavily_key
```

MCP workflows run in a normal Python subprocess to avoid the Windows Jupyter subprocess issue.


In [5]:
import os
import subprocess
import sys
import tempfile
import textwrap

from dotenv import load_dotenv

load_dotenv(override=True)

def run_mcp(code_text, timeout=900):
    """Run MCP code with the same Python interpreter as this notebook."""
    with tempfile.NamedTemporaryFile(
        mode="w",
        suffix=".py",
        delete=False,
        encoding="utf-8",
        dir=os.getcwd(),
    ) as f:
        f.write(textwrap.dedent(code_text))
        script = f.name

    try:
        result = subprocess.run(
            [sys.executable, script],
            capture_output=True,
            text=True,
            timeout=timeout,
            env=os.environ.copy(),
        )

        if result.stdout:
            print(result.stdout)

        if result.stderr:
            print(result.stderr)

        if result.returncode != 0:
            raise RuntimeError(f"MCP process failed: {result.returncode}")
    finally:
        try:
            os.remove(script)
        except OSError:
            pass

print("Python:", sys.executable)
print("OPENAI_API_KEY found:", bool(os.getenv("OPENAI_API_KEY")))
print("TAVILY_API_KEY found:", bool(os.getenv("TAVILY_API_KEY")))


Python: c:\Users\Sealion\Desktop\sos_2026 Study_new\sos agents-AI-main\.venv\Scripts\python.exe
OPENAI_API_KEY found: True
TAVILY_API_KEY found: True


## 2. Research Agent: Tavily → Qdrant

The **Biomedical Research Agent** has access to both Tavily and Qdrant.

Its job is to:

- perform multiple real-time searches,
- extract useful source content,
- prioritize authoritative biomedical sources,
- reject weak or duplicate claims,
- store the strongest evidence in Qdrant.

Preferred sources:

1. ClinicalTrials.gov
2. NCI / NIH
3. PubMed and peer-reviewed journals
4. major academic cancer centers
5. other reputable medical sources


In [6]:
research_script = r"""
import asyncio
import os
import sys
from datetime import date
from pathlib import Path

from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio


# Tavily also exposes tavily_research, an agentic deep-research tool that runs for
# many minutes. One call to it outlives the MCP session timeout and fails the whole
# run, so the agent is given only the fast tools.
# Note: this script lives inside a raw triple-quoted string, so it must not
# contain a nested triple quote anywhere, including in docstrings or comments.
class FilteredMCPServer:

    def __init__(self, server, allowed_names):
        self._server = server
        self._allowed = set(allowed_names)

    async def list_tools(self):
        tools = await self._server.list_tools()
        return [t for t in tools if t.name in self._allowed]

    async def call_tool(self, tool_name, tool_input):
        return await self._server.call_tool(tool_name, tool_input)

    def __getattr__(self, name):
        return getattr(self._server, name)


MODEL = "gpt-5.4-mini"

TAVILY_TOOLS = ["tavily_search", "tavily_extract"]

QUESTION = "What are the latest mRNA-based cancer treatments?"


async def main():

    # Get Tavily API key from .env
    tavily_key = os.getenv("TAVILY_API_KEY")

    if not tavily_key:
        raise RuntimeError(
            "TAVILY_API_KEY is missing from .env"
        )


    # ----------------------------------------
    # Tavily MCP
    # Real-time web search + content extraction
    # ----------------------------------------

    tavily_params = {
        "command": "npx.cmd" if sys.platform == "win32" else "npx",

        "args": [
            "-y",
            "tavily-mcp@latest",
        ],

        "env": {
            **os.environ,

            "NODE_OPTIONS": "--use-system-ca",

            "TAVILY_API_KEY": tavily_key,
        },
    }


    # ----------------------------------------
    # Qdrant local storage
    # ----------------------------------------

    qdrant_path = Path(
        "memory/mrna_cancer_qdrant"
    ).resolve()

    cache_path = Path(
        "memory/fastembed"
    ).resolve()


    qdrant_path.mkdir(
        parents=True,
        exist_ok=True,
    )

    cache_path.mkdir(
        parents=True,
        exist_ok=True,
    )


    # ----------------------------------------
    # Qdrant MCP
    # ----------------------------------------

    qdrant_params = {
        "command": "uvx",

        "args": [
            "--system-certs",
            "mcp-server-qdrant",
        ],

        "env": {
            **os.environ,

            "QDRANT_LOCAL_PATH": str(
                qdrant_path
            ),

            "COLLECTION_NAME": (
                "mrna_cancer_evidence"
            ),

            "FASTEMBED_CACHE_PATH": str(
                cache_path
            ),
        },
    }


    # ----------------------------------------
    # Research Agent instructions
    # ----------------------------------------

    instructions = (
        "You are a biomedical research and "
        "knowledge-curation agent. "

        "Use tavily_search for web research and "
        "tavily_extract to read a promising page. "

        "Run several narrow searches rather than one "
        "broad one, and keep each search focused on a "
        "single cancer type, trial, or treatment. "

        "Prefer ClinicalTrials.gov, NCI/NIH, "
        "PubMed, peer-reviewed journals, and "
        "major academic cancer centers. "

        "Focus on personalized mRNA neoantigen "
        "vaccines, melanoma, pancreatic cancer, "
        "other cancers with meaningful recent "
        "evidence, checkpoint inhibitor combinations, "
        "mechanism, trial phase, efficacy, safety, "
        "manufacturing, access, and uncertainties. "

        "For each strong evidence item, preserve "
        "cancer type, treatment name, combination "
        "therapy, trial phase, key result, date, "
        "limitations, source name, and source URL. "

        "Store about 8-12 concise evidence chunks "
        "in Qdrant. "

        "Do not store duplicate, promotional, weak, "
        "or unsupported claims. "

        "Clearly distinguish late-stage evidence "
        "from early-stage or investigational evidence."
    )


    # ----------------------------------------
    # Research task
    # ----------------------------------------

    task = (
        "Research this question:\n\n"

        + QUESTION

        + "\n\n"

        + "Use evidence current through "

        + date.today().isoformat()

        + ".\n\n"

        + "Search multiple times if needed. "

        + "Evaluate source quality, extract useful "
          "evidence, and store the strongest findings "
          "in Qdrant."
    )


    # ----------------------------------------
    # Start Tavily MCP + Qdrant MCP
    # ----------------------------------------

    async with MCPServerStdio(
        params=tavily_params,
        client_session_timeout_seconds=300,

    ) as tavily_server, MCPServerStdio(

        params=qdrant_params,
        client_session_timeout_seconds=300,

    ) as qdrant_server:


        # ----------------------------------------
        # Hide tavily_research from the agent
        # ----------------------------------------

        search_server = FilteredMCPServer(
            tavily_server,
            TAVILY_TOOLS,
        )


        # ----------------------------------------
        # Create Biomedical Research Agent
        # ----------------------------------------

        agent = Agent(
            name="biomedical_research_agent",

            instructions=instructions,

            model=MODEL,

            mcp_servers=[
                search_server,
                qdrant_server,
            ],
        )


        # ----------------------------------------
        # Run the research workflow
        # ----------------------------------------

        with trace(
            "Tavily research to Qdrant"
        ):

            result = await Runner.run(
                agent,
                task,
                max_turns=30,
            )


        # ----------------------------------------
        # Show research-agent result
        # ----------------------------------------

        print(
            result.final_output
        )


    await asyncio.sleep(0.5)


# ----------------------------------------
# Windows entry point
# ----------------------------------------

if __name__ == "__main__":

    if sys.platform == "win32":

        asyncio.set_event_loop_policy(
            asyncio.WindowsProactorEventLoopPolicy()
        )

    asyncio.run(
        main()
    )
"""


# ----------------------------------------
# Run from the Jupyter notebook
# ----------------------------------------

run_mcp(
    research_script, 
    timeout=3000, 
)


Hereâ€™s the current evidence-based picture of the latest **mRNA-based cancer treatments** through **2026-08-27**.

## Bottom line
The most advanced and clinically convincing mRNA-based cancer treatments are still **personalized neoantigen vaccines**, almost always used **with checkpoint inhibitors**:

1. **Melanoma:** **intismeran autogene** (formerly **mRNA-4157/V940**) + **pembrolizumab**  
   - Best late-stage evidence so far
   - Randomized **phase 2b** data with durable recurrence-free survival benefit
   - Phase 3 confirmation is underway

2. **Pancreatic ductal adenocarcinoma (PDAC):** **autogene cevumeran** (BNT122/RO7198457)  
   - Strong early translational and clinical signal
   - Best evidence is from a small **phase 1** study with durable T-cell responses and longer recurrence-free survival in responders
   - A randomized **phase 2** adjuvant trial is ongoing

3. **Other cancers:** several **early-phase investigational** personalized mRNA vaccine programs are now in trial

## 3. Oncology Agentic RAG: Retrieve from Qdrant

The **Oncology RAG Agent** now uses only Qdrant.

It does not search the web again.

This makes the RAG stage clear:

```text
Tavily research
      ↓
Curated evidence
      ↓
Qdrant
      ↓
Agentic retrieval
      ↓
Grounded synthesis
```

The agent decides what evidence it needs, retrieves relevant chunks, compares evidence strength, and identifies uncertainty.


In [7]:
rag_script = r"""
import asyncio
import os
import sys
from pathlib import Path

from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio


MODEL = "gpt-5.4-mini"

QUESTION = "What are the latest mRNA-based cancer treatments?"


async def main():

    # ----------------------------------------
    # Qdrant MCP
    # Same path and collection as the research
    # step, so we read back what it stored.
    # ----------------------------------------

    qdrant_params = {
        "command": "uvx",

        "args": [
            "--system-certs",
            "mcp-server-qdrant",
        ],

        "env": {
            **os.environ,

            "QDRANT_LOCAL_PATH": str(
                Path("memory/mrna_cancer_qdrant").resolve()
            ),

            "COLLECTION_NAME": (
                "mrna_cancer_evidence"
            ),

            "FASTEMBED_CACHE_PATH": str(
                Path("memory/fastembed").resolve()
            ),
        },
    }


    # ----------------------------------------
    # RAG Agent instructions
    # No web access here, so every claim has to
    # come out of the vector store.
    # ----------------------------------------

    instructions = (
        "You are an Oncology Agentic RAG assistant. "

        "Answer only from evidence retrieved from "
        "Qdrant. "

        "Search the vector knowledge base multiple "
        "times when useful. "

        "Clearly distinguish late-stage evidence "
        "from early-stage evidence. "

        "Do not overstate preliminary findings. "

        "Preserve source names and URLs. "

        "Organize the answer into: Overview; "
        "How personalized mRNA cancer vaccines work; "
        "Strongest current clinical evidence; "
        "Melanoma; "
        "Pancreatic cancer; "
        "Other notable cancer research; "
        "Why checkpoint inhibitors are combined with "
        "mRNA vaccines; "
        "Safety and practical limitations; "
        "What remains uncertain; "
        "Bottom line."
    )


    # ----------------------------------------
    # Retrieval task
    # ----------------------------------------

    task = (
        "Answer this question using only the "
        "Qdrant knowledge base:\n\n"

        + QUESTION

        + "\n\n"

        + "Focus on the most recent and clinically "
          "important evidence. "

        + "Explain which approaches have the strongest "
          "evidence, which remain investigational, and "
          "what still needs to be proven before broad "
          "clinical use."
    )


    # ----------------------------------------
    # Start Qdrant MCP only
    # ----------------------------------------

    async with MCPServerStdio(
        params=qdrant_params,
        client_session_timeout_seconds=300,

    ) as qdrant_server:


        # ----------------------------------------
        # Create Oncology RAG Agent
        # ----------------------------------------

        agent = Agent(
            name="oncology_rag_agent",

            instructions=instructions,

            model=MODEL,

            mcp_servers=[
                qdrant_server,
            ],
        )


        # ----------------------------------------
        # Run the retrieval workflow
        # ----------------------------------------

        with trace(
            "Oncology Agentic RAG"
        ):

            result = await Runner.run(
                agent,
                task,
                max_turns=25,
            )


        # ----------------------------------------
        # Show grounded synthesis
        # ----------------------------------------

        print(
            result.final_output
        )


    await asyncio.sleep(0.5)


# ----------------------------------------
# Windows entry point
# ----------------------------------------

if __name__ == "__main__":

    if sys.platform == "win32":

        asyncio.set_event_loop_policy(
            asyncio.WindowsProactorEventLoopPolicy()
        )

    asyncio.run(
        main()
    )
"""


# ----------------------------------------
# Run from the Jupyter notebook
# ----------------------------------------

run_mcp(
    rag_script,
    timeout=3600,
)

## Overview
The **latest mRNA-based cancer treatments** in the knowledge base are mainly **personalized mRNA neoantigen vaccines**. The strongest evidence is in:

1. **Melanoma** â€” including a **phase III** report of reduced recurrence risk with a personalized mRNA vaccine, described as the **first mRNA-based cancer treatment to show success in a late-stage trial**.  
2. **Pancreatic cancer** â€” early-phase but biologically promising data showing personalized RNA neoantigen vaccines can prime **long-lived CD8+ T cells** and may be associated with better outcomes in responders.

Outside these, most other mRNA cancer approaches remain **investigational** or supported mainly by reviews, trial listings, or early clinical studies.

## How personalized mRNA cancer vaccines work
According to the National Cancer Institute Drug Dictionary, the platform works like this:

- **Tumor sequencing** identifies **tumor-specific neoantigens**
- Selected neoantigen sequences are **encoded in mRNA**
- 

## 4. Why This Is Agentic RAG

```text
Research Question
        ↓
Biomedical Research Agent
        ↓
Plans Tavily searches
        ↓
Extracts and evaluates evidence
        ↓
Selects what to store
        ↓
Qdrant knowledge base
        ↓
Oncology RAG Agent
        ↓
Plans retrieval
        ↓
Runs semantic searches
        ↓
Compares evidence strength
        ↓
Identifies uncertainty
        ↓
Produces grounded synthesis
```

The first agent actively **builds knowledge**.

The second agent actively **retrieves and uses knowledge**.

That makes this a stronger biomedical Agentic RAG project than a fixed evidence snapshot.


## 5. Concepts Demonstrated

- Agentic RAG
- Tavily MCP
- real-time web search
- content extraction
- biomedical source prioritization
- evidence curation
- Qdrant vector database
- semantic retrieval
- multi-step retrieval
- clinical evidence comparison
- uncertainty-aware synthesis
- grounded LLM generation
- OpenAI Agents SDK
- MCP orchestration
